# 🏦 Kaggle Pipeline 2 — Resume Training + Eval

Notebook này tiếp tục training từ checkpoint của Session 1:
1. Copy checkpoint từ dataset đã lưu
2. Resume training (epochs còn lại)
3. Evaluate trên test set
4. Package artifacts + zip output

**Yêu cầu:** Đã add dataset `banking-checkpoint` (output của Session 1) vào notebook này.

In [ ]:
# === 1) Config ===
GITHUB_USERNAME = "tzin1401"
REPO_NAME = "banking-intent-unsloth"
BRANCH = "main"
TRAIN_CONFIG = "configs/train_3b_t4x2.yaml"

# Đường dẫn đến checkpoint từ Session 1
# Sửa nếu tên dataset khác
PREV_OUTPUT = "/kaggle/input/banking-checkpoint"

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
print("Repo:", REPO_URL)
print("Checkpoint source:", PREV_OUTPUT)

In [ ]:
# === 2) Install dependencies ===
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton -q
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "transformers<=5.5.0" "datasets<4.4.0" "dill>=0.3.9" pyyaml scikit-learn -q
print("\n✅ Dependencies installed")

In [ ]:
# === 3) Clone repo ===
import os

if not os.path.exists(REPO_NAME):
    !git clone "$REPO_URL"

%cd "$REPO_NAME"
!git fetch origin
!git checkout "$BRANCH"
!git pull origin "$BRANCH"
!git rev-parse --short HEAD

In [ ]:
# === 4) Copy checkpoint + data từ Session 1 ===
import os, shutil, glob

# Tìm đường dẫn chính xác trong dataset
# Kaggle output có thể nằm ở nhiều đường dẫn khác nhau
possible_paths = [
    f"{PREV_OUTPUT}/banking-intent-unsloth",
    f"{PREV_OUTPUT}/kaggle/working/banking-intent-unsloth",
    f"{PREV_OUTPUT}",
]

src_root = None
for p in possible_paths:
    if os.path.exists(os.path.join(p, "outputs")):
        src_root = p
        break

if src_root is None:
    print("❌ Không tìm thấy outputs/ trong dataset. Kiểm tra lại:")
    !find {PREV_OUTPUT} -maxdepth 4 -type d -name outputs
    raise FileNotFoundError("Không tìm thấy checkpoint")

print(f"✅ Tìm thấy checkpoint tại: {src_root}")

# Copy outputs (checkpoint) và sample_data
!cp -r "{src_root}/outputs" ./outputs
!cp -r "{src_root}/sample_data" ./sample_data

# Liệt kê checkpoints
print("\n📂 Checkpoints có sẵn:")
!ls -d outputs/checkpoint-* 2>/dev/null || echo "Không có checkpoint dir"
print("\n📂 Sample data:")
!ls sample_data/

In [ ]:
# === 5) Enable resume + Train ===
import os, yaml

os.environ["TRAIN_CONFIG"] = TRAIN_CONFIG

# Đọc config và bật resume_from_checkpoint
with open(TRAIN_CONFIG, "r") as f:
    config = yaml.safe_load(f)

config["resume_from_checkpoint"] = True

with open(TRAIN_CONFIG, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"✅ Config: resume_from_checkpoint = True")
print(f"   Total epochs: {config['num_train_epochs']}")
print(f"   Sẽ tự động skip epochs đã hoàn thành\n")

# Train!
!python scripts/train.py

In [ ]:
# === 6) Package artifacts ===
!python scripts/package_artifacts.py

In [ ]:
# === 7) Preview results ===
print("=" * 50)
!cat outputs/eval_results.txt
print("=" * 50)

!cat artifacts/LATEST.txt
latest = open("artifacts/LATEST.txt", "r").read().strip()
print(f"\nLatest run: {latest}")
!ls -lah "artifacts/{latest}"

In [ ]:
# === 8) Zip tất cả output để tải về ===
!zip -r /kaggle/working/banking_3b_final.zip outputs/ sample_data/ configs/ artifacts/
print("\n✅ File zip sẵn sàng tải về: banking_3b_final.zip")
print("   → Vào Output tab của notebook để download.")